# HEA Hardness KAN — Optuna Hyperparameter Optimisation

This notebook shows how to run an Optuna Bayesian search over KAN hyperparameters
for a chosen hidden-node count, using the same 5-fold cross-validation routine
as the main study.

---

## Already have optimised parameters?

Best hyperparameters from a 100-trial Optuna sweep (fixed shuffle seed = 42)
are already stored in:

```
HEA-KANs/best_hyperparams_by_hidden_nodes.json
```

To load and use them directly — without running this notebook — see
`HEA-pyKAN-demo.ipynb`, which walks through loading, parameter extraction,
and cross-validated evaluation.

---

## A note on what "optimised" means here

With only 244 samples, each cross-validation run is inherently noisy: the
same model trained on two different random splits of the data can easily give
MAE values that differ by 5–10 HV.  Optuna will find a parameter set that
performed well on the particular set of folds it saw during the search, but
this is not guaranteed to be the globally best set — it is better described
as *a reasonable set that is unlikely to be badly wrong*.

Practically this means:
- More trials help, but with diminishing returns beyond ~50–100 on a dataset this size.
- Fixing the shuffle seed (recommended) makes trials comparable within a study,
  but the best trial found may not generalise perfectly to other splits.
- The repeated CV in `HEA-pyKAN-demo.ipynb` gives a more honest performance
  estimate for any given parameter set than the Optuna trial value alone.

---

## Parameters being optimised

| Parameter | Type | Range | Notes |
|---|---|---|---|
| `k` | int | 1 – 6 | Spline order |
| `g` | int | 1 – 6 | Spline grid intervals |
| `lamb` | float (log) | 1×10⁻⁷ – 0.1 | Master regularisation weight |
| `lamb_entropy_actual` | float (log) | 1×10⁻⁷ – 1 | Absolute entropy penalty weight |
| `model_seed` | categorical | 0 – 9 | KAN weight initialisation seed |

`lamb_entropy_actual` is divided by `lamb` before being passed to the model
(see regularisation note in `HEA-pyKAN-demo.ipynb`).  The *absolute* value
is stored in the JSON so results are interpretable without needing to know `lamb`.

In [ ]:
import json
import sys
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import optuna
from kan import *

sys.path.insert(0, os.path.abspath(''))
from HEA_pyKAN import k_fold_val, get_best_result

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

## 1  Load and normalise data

Same preprocessing as `HEA-pyKAN-demo.ipynb` — z-score normalise all features
and retain `stdev["HV"]` to convert model outputs back to HV units.

In [ ]:
DATA_PATH = r"H:\My Drive\Documents and Data\Data\BerryML"
DATA_FILE = "Gorsse2018ML_Data_RawHV.csv"

df   = pd.read_csv(os.path.join(DATA_PATH, DATA_FILE))
cols = list(df.columns)
df   = df[df["HV"].notna()][cols[25:]]

mean    = df.mean()
stdev   = df.std()
norm_df = (df - mean) / stdev

x_data = torch.tensor(norm_df[cols[26:]].values, dtype=torch.float32).to(device)
y_data = torch.tensor(norm_df[cols[25]].values,  dtype=torch.float32).unsqueeze(-1).to(device)

data_norm = stdev["HV"]
print(f"x: {x_data.shape},  y: {y_data.shape}  |  HV std = {data_norm:.1f}")

## 2  Define the Optuna objective

`TuneObjective` is defined inline here rather than imported from `HEA_pyKAN`.
This is intentional — the objective is tightly coupled to the Optuna workflow
and the specific parameter ranges being searched, so it lives in the
optimisation notebook rather than the general library.  If you want to
customise the search ranges or add new parameters, edit the class below.

Each Optuna trial proposes a set of hyperparameters, trains models via
`k_fold_val`, and returns the mean cross-validated MAE (in HV units) as the
value to minimise.

A fixed `shuffle_seed` is passed to `k_fold_val` so that every trial sees
the same fold splits.  This is important: without it, a trial might appear
better simply because it drew an easier split, not because its hyperparameters
are genuinely superior.

In [ ]:
class TuneObjective:
    """Optuna objective for a fixed hidden-node count."""

    def __init__(self, hidden, x_data, y_data, data_norm, splits=5, shuffle_seed=None):
        self.hidden      = hidden
        self.x_data      = x_data
        self.y_data      = y_data
        self.data_norm   = data_norm
        self.splits      = splits
        self.shuffle_seed = shuffle_seed

    def __call__(self, trial):

        k    = trial.suggest_int(  "k",    1, 6)
        g    = trial.suggest_int(  "g",    1, 6)
        lamb = trial.suggest_float("lamb", 1e-7, 1e-1, log=True)

        # optimise the *absolute* entropy weight, then convert to model parameter
        lamb_entropy_actual = trial.suggest_float("lamb_entropy", 1e-7, 1.0, log=True)
        lamb_entropy        = lamb_entropy_actual / lamb

        model_seed = trial.suggest_categorical("model_seed", list(range(10)))

        mae, mae_error, stddev = k_fold_val(
            self.x_data,
            self.y_data,
            data_norm    = self.data_norm,
            model_seed   = model_seed,
            shuffle_seed = self.shuffle_seed,
            splits       = self.splits,
            hidden       = self.hidden,
            grid         = g,
            k            = k,
            steps        = 25,
            loss_fn      = nn.L1Loss(),
            lamb         = lamb,
            lamb_entropy = lamb_entropy,
            device       = str(device),
            verbose      = False,
        )

        # store extra info accessible via trial.user_attrs
        trial.set_user_attr("error",  mae_error)
        trial.set_user_attr("stddev", stddev)
        trial.set_user_attr("edges",  11 * self.hidden)
        trial.set_user_attr("hidden", self.hidden)

        return mae

## 3  Configure and run the study

The study is backed by a SQLite database so that:
- Progress is saved after every trial — you can interrupt and resume without losing work.
- Multiple processes could run trials in parallel against the same DB (Optuna handles the locking).
- The full trial history is available for post-hoc analysis.

Change `HIDDEN` and `TARGET_TRIALS` to taste.  Running the cell again on an
existing database will load the study and add only the remaining trials needed
to reach `TARGET_TRIALS`.

In [ ]:
HIDDEN        = 20    # hidden-node count to optimise for
TARGET_TRIALS = 100   # total trials to run (resumes if DB already exists)
SHUFFLE_SEED  = 42    # fixed so all trials see the same fold splits

RESULTS_DIR = r"H:\My Drive\Documents and Data\Data\BerryML\my_optuna_sweep"
os.makedirs(RESULTS_DIR, exist_ok=True)

db_file      = os.path.join(RESULTS_DIR, f"study_hidden_{HIDDEN}.db")
storage_url  = f"sqlite:///{db_file}"
study_name   = f"hidden_{HIDDEN}"

# load existing study or create a new one
try:
    study = optuna.load_study(study_name=study_name, storage=storage_url)
    print(f"Loaded existing study ({len(study.trials)} trials completed)")
except KeyError:
    study = optuna.create_study(study_name=study_name, direction="minimize", storage=storage_url)
    print("Created new study")

remaining = max(0, TARGET_TRIALS - len(study.trials))
if remaining > 0:
    print(f"Running {remaining} trials...")
    optuna.logging.set_verbosity(optuna.logging.WARNING)  # suppress per-trial spam
    study.optimize(
        TuneObjective(HIDDEN, x_data, y_data, data_norm, shuffle_seed=SHUFFLE_SEED),
        n_trials=remaining,
        show_progress_bar=True,
    )
else:
    print("Target already reached — skipping optimisation")

## 4  Inspect results

The best trial's parameters are printed below.  Bear in mind the caveats from
the introduction: the best trial is the one that happened to perform well on
these particular folds.  Use `HEA-pyKAN-demo.ipynb` with repeated CV to get a
more robust estimate of how these parameters actually generalise.

In [ ]:
best = get_best_result(study)

print(f"Best trial for hidden={HIDDEN}")
print(f"  CV MAE            : {best['meanHV']:.2f} ± {best['error']:.2f} HV")
print(f"  Stdev across folds: {best['stddev']:.2f} HV")
print()
print("  Parameters:")
for key, val in best["params"].items():
    print(f"    {key:<20} = {val}")

# show the lamb_entropy conversion for clarity
p = best["params"]
print()
print(f"  lamb_entropy (stored / absolute) = {p['lamb_entropy']:.3e}")
print(f"  lamb_entropy (model parameter)   = {p['lamb_entropy']/p['lamb']:.3e}")

In [ ]:
# trial history — useful for checking whether the search has converged
trial_values = [t.value for t in study.trials if t.value is not None]
running_best = np.minimum.accumulate(trial_values)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].scatter(range(len(trial_values)), trial_values, s=10, alpha=0.5, label="trial MAE")
axes[0].plot(running_best, color="crimson", lw=1.5, label="running best")
axes[0].set_xlabel("Trial")
axes[0].set_ylabel("CV MAE (HV)")
axes[0].set_title(f"Optimisation history  |  hidden={HIDDEN}")
axes[0].legend()

axes[1].hist(trial_values, bins=20, color="steelblue", alpha=0.8, edgecolor="white")
axes[1].axvline(study.best_value, color="crimson", lw=1.5, linestyle="--",
                label=f"best = {study.best_value:.1f} HV")
axes[1].set_xlabel("CV MAE (HV)")
axes[1].set_ylabel("Count")
axes[1].set_title("Distribution of trial values")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nTotal trials: {len(trial_values)}")
print(f"Best value reached at trial {np.argmin(trial_values)}")
print("If the running-best curve has plateaued, more trials are unlikely to help much.")

## 5  Save results

Save the best parameters to JSON in the same format as
`best_hyperparams_by_hidden_nodes.json` so they can be loaded directly by
`HEA-pyKAN-demo.ipynb`.

If you run this optimisation for multiple hidden-node counts, you can merge
all results into a single JSON (see the commented block below).

In [ ]:
t = study.best_trial
result_entry = {
    str(HIDDEN): {
        "value":      t.value,
        "params":     t.params,
        "user_attrs": t.user_attrs,
    }
}

out_file = os.path.join(RESULTS_DIR, f"best_params_hidden_{HIDDEN}.json")
with open(out_file, "w") as f:
    json.dump(result_entry, f, indent=2)

print(f"Saved to {out_file}")
print()
print("To use in HEA-pyKAN-demo.ipynb, point HYPERPARAM_FILE at this file,")
print("or merge entries into best_hyperparams_by_hidden_nodes.json.")

In [ ]:
# ── Optional: merge results from multiple hidden-node studies into one JSON ──
#
# Run the cells above for each HIDDEN value, then run this cell to combine.
#
# all_results = {}
# for h in [1, 2, 3, 4, 6, 8, 10, 15, 20, 25, 30]:
#     single_file = os.path.join(RESULTS_DIR, f"best_params_hidden_{h}.json")
#     if os.path.exists(single_file):
#         with open(single_file) as f:
#             all_results.update(json.load(f))
#
# merged_file = os.path.join(RESULTS_DIR, "best_hyperparams_by_hidden_nodes.json")
# with open(merged_file, "w") as f:
#     json.dump(all_results, f, indent=2)
# print(f"Merged {len(all_results)} entries → {merged_file}")